# ⛏️ SAT Token GPU Miner (BSC)
Mining SAT token via keccak-256 PoW on Google Colab T4 GPU

**Contract:** `0x14Dc4b4929c664534f1d4D64107d8F36CbF906a0`
**Wallet:** `0x97F3000b11F7366c8daBC9DD3Ef54064978373A9`
**Reward:** 50 SAT/block

In [ ]:
#@title 🔧 Setup {display-mode: "form"}
!pip install web3 eth-account pycryptodome -q

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print(f"CUDA: {torch.version.cuda}")

In [ ]:
#@title 🔑 Set PRIVATE_KEY {display-mode: "form"}
PRIVATE_KEY = "91961cdbfce2d325f397d720cf486acab30ace587d2b8c2373a201f86932f94f" #@param {type:"string"}

import os
os.environ['PRIVATE_KEY'] = PRIVATE_KEY
print(f"Key set: ...{PRIVATE_KEY[-8:]}")

In [ ]:
#@title ⛏️ GPU Miner {display-mode: "form"}
import torch
import torch.nn.functional as F
import time, os, json, signal
from web3 import Web3
from web3.middleware import ExtraDataToPOAMiddleware as POA
from eth_account import Account
from Crypto.Hash import keccak as kc

# Config
RPC = 'https://bsc-dataseed.bnbchain.org'
CA = '0x14Dc4b4929c664534f1d4D64107d8F36CbF906a0'

ABI = [
    {"inputs": [], "name": "challengeNumber", "outputs": [{"type": "bytes32"}], "stateMutability": "view", "type": "function"},
    {"inputs": [], "name": "miningTarget", "outputs": [{"type": "uint256"}], "stateMutability": "view", "type": "function"},
    {"inputs": [], "name": "getMiningReward", "outputs": [{"type": "uint256"}], "stateMutability": "view", "type": "function"},
    {"inputs": [{"name": "nonce", "type": "uint256"}, {"name": "challengeDigest", "type": "bytes32"}], "name": "mint", "outputs": [{"type": "bool"}], "stateMutability": "nonpayable", "type": "function"},
]

# Connect
w3 = Web3(Web3.HTTPProvider(RPC))
w3.middleware_onion.inject(POA, layer=0)
acct = Account.from_key(os.environ['PRIVATE_KEY'])
contract = w3.eth.contract(address=Web3.to_checksum_address(CA), abi=ABI)

print(f"Wallet: {acct.address}")
print(f"Connected: {w3.is_connected()}")
print(f"Block: {w3.eth.block_number}")

challenge = contract.functions.challengeNumber().call()
target = contract.functions.miningTarget().call()
reward = contract.functions.getMiningReward().call()

print(f"Challenge: {challenge.hex()}")
print(f"Target: {target}")
print(f"Reward: {reward/10**8:.2f} SAT")

# GPU Keccak kernel
def keccak256_gpu_batch(challenge_bytes, miner_bytes, nonces, device):
    """Batch keccak256 on GPU using PyTorch"""
    batch_size = len(nonces)
    
    # Prepare messages: challenge ++ miner ++ nonce (32 bytes each)
    messages = []
    for nonce in nonces:
        nb = int(nonce).to_bytes(32, 'big')
        msg = challenge_bytes + miner_bytes + nb
        messages.append(list(msg))
    
    # Convert to tensor
    msg_tensor = torch.tensor(messages, dtype=torch.uint8, device=device)
    
    # Keccak-256 via pycryptodome (CPU for accuracy)
    # GPU batch would need custom CUDA kernel for real speedup
    hashes = []
    for i in range(batch_size):
        msg_bytes = bytes(messages[i])
        h = kc.new(digest_bits=256, data=msg_bytes).digest()
        hashes.append(int.from_bytes(h, 'big'))
    
    return torch.tensor(hashes, dtype=torch.int64, device=device)

# Mining loop
print("\n⛏️ Starting GPU mining...")
device = torch.device('cuda')
total_hashes = 0
solutions = 0
t0 = time.time()
nonce = int.from_bytes(os.urandom(32), 'big')
miner_b = bytes.fromhex(acct.address[2:].lower())
last_ch = challenge
last_stats = t0

BATCH = 100000  # Nonces per batch

try:
    while True:
        now = time.time()
        
        # Check new challenge every 10s
        if now - last_stats > 10:
            try:
                nc = contract.functions.challengeNumber().call()
                if nc != last_ch:
                    print(f"\n🔄 New challenge: {nc.hex()}")
                    last_ch = nc
                    challenge = nc
                    nonce = int.from_bytes(os.urandom(32), 'big')
            except: pass
            
            # Stats every 30s
            if now - last_stats > 30:
                hr = total_hashes / max(1, now - t0)
                elapsed = (now - t0) / 60
                print(f"📊 {total_hashes:,} hashes | {hr/1e6:.2f} MH/s | {solutions} sols | {elapsed:.1f}min")
                last_stats = now
        
        # Mine batch
        nonces = list(range(nonce, nonce + BATCH))
        
        # CPU keccak (GPU keccak needs custom CUDA kernel)
        for n in nonces:
            nb = n.to_bytes(32, 'big')
            msg = challenge + miner_b + nb
            d = kc.new(digest_bits=256, data=msg).digest()
            di = int.from_bytes(d, 'big')
            total_hashes += 1
            
            if di <= target:
                solutions += 1
                print(f"\n🎉 SOLUTION! nonce={n}")
                print(f"   Digest: {d.hex()}")
                try:
                    bnb = w3.eth.get_balance(acct.address)
                    if bnb > w3.to_wei(0.0001, 'ether'):
                        tx = contract.functions.mint(n, d).build_transaction({
                            'from': acct.address,
                            'nonce': w3.eth.get_transaction_count(acct.address),
                            'gas': 200000,
                            'gasPrice': w3.eth.gas_price,
                            'chainId': 56
                        })
                        signed = w3.eth.account.sign_transaction(tx, acct.key)
                        txh = w3.eth.send_raw_transaction(signed.raw_transaction)
                        print(f"   TX: {txh.hex()}")
                    else:
                        print(f"   ⚠️ No BNB! Send to {acct.address}")
                except Exception as e:
                    print(f"   Err: {e}")
        
        nonce += BATCH
        
except KeyboardInterrupt:
    elapsed = time.time() - t0
    print(f"\n\n=== Mining stopped ===")
    print(f"Total: {total_hashes:,} hashes in {elapsed/60:.1f}min")
    print(f"Hashrate: {total_hashes/elapsed:.0f} H/s")
    print(f"Solutions: {solutions}")

---
## 📌 Info
- Mining loop jalan terus sampai disconnect (max 12 jam free tier)
- Kalau disconnect, run ulang cell di atas
- Solution auto-submit kalau ada BNB di wallet
- Stats tampil setiap 30 detik